<a href="https://colab.research.google.com/github/miriamamin1213-ux/FinalModels/blob/main/GPT2Tab_Hecktor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# GPT-2 Numerical Tabular - HECKTOR - Seed 42 - No SMOTE
# ============================================================

!pip install -q gdown

import os,random,numpy as np,pandas as pd
from torch.utils.data import TensorDataset,DataLoader
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OrdinalEncoder
from sklearn.metrics import classification_report,balanced_accuracy_score,roc_auc_score,f1_score
from transformers import GPT2Model
import gdown

# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"]=str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True,warn_only=True)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False

SEED=42
seed_everything(SEED)

# ============================================================
# Device
# ============================================================

device=torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:",device)

# ============================================================
# Load HECKTOR data
# ============================================================

gdown.download(
    id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",
    output="HPV2025.xlsx",
    quiet=False
)

df=pd.read_excel("HPV2025.xlsx")

df=df.dropna(
    subset=["HPV Status"]
)

df=df.drop(
    columns=[
        "PatientID",
        "CenterID",
        "Task 1",
        "Task 2",
        "Task 3"
    ]
)

# ============================================================
# Stage Conversion
# ============================================================

df["T-stage"]=df["T-stage"].replace({
    "T0":0,
    "T1":1,
    "T2":2,
    "T3":3,
    "T4":4
})

df["N-stage"]=df["N-stage"].replace({
    "N0":0,
    "N1":1,
    "N2":2,
    "N3":3
})

df["M-stage"]=df["M-stage"].replace({
    "M0":0,
    "M1":1
})

# ============================================================
# Features and Target
# ============================================================

feature_columns=[
    "Age",
    "Gender",
    "Tobacco Consumption",
    "Alcohol Consumption",
    "Performance Status",
    "Relapse",
    "RFS",
    "Treatment",
    "T-stage",
    "N-stage",
    "M-stage"
]

X=df[feature_columns].copy()
y=df["HPV Status"].copy()

print("Dataset shape:",df.shape)

# ============================================================
# Stratified train/test split
# ============================================================

X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Train samples:")
print(y_train.value_counts())

print("\nTest samples:")
print(y_test.value_counts())

# ============================================================
# Train-only preprocessing
# ============================================================

categorical_features=[
    c for c in X_train.columns
    if X_train[c].dtype==object
]

numerical_features=[
    c for c in X_train.columns
    if c not in categorical_features
]

# ============================================================
# Train-only imputation
# ============================================================

for col in categorical_features:
    value=X_train[col].mode().iloc[0]
    X_train[col]=X_train[col].fillna(value)
    X_test[col]=X_test[col].fillna(value)

for col in numerical_features:
    value=X_train[col].median()
    X_train[col]=X_train[col].fillna(value)
    X_test[col]=X_test[col].fillna(value)

# ============================================================
# Train-only label encoding
# ============================================================

encoder=OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

if categorical_features:

    X_train[categorical_features]=encoder.fit_transform(
        X_train[categorical_features]
    )

    X_test[categorical_features]=encoder.transform(
        X_test[categorical_features]
    )

# ============================================================
# StandardScaler - fit on train only
# ============================================================

scaler=StandardScaler()

X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

# ============================================================
# No SMOTE
# ============================================================

print("\nTraining samples without SMOTE:")
print(pd.Series(y_train).value_counts())

# ============================================================
# Convert to tensors
# ============================================================

X_train=torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train=torch.tensor(
    np.asarray(y_train),
    dtype=torch.long
)

X_test=torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test=torch.tensor(
    np.asarray(y_test),
    dtype=torch.long
)

# ============================================================
# DataLoader
# ============================================================

BATCH_SIZE=256

train_dataset=TensorDataset(
    X_train,
    y_train
)

train_generator=torch.Generator()
train_generator.manual_seed(SEED)

train_loader=DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# ============================================================
# GPT-2 model for tabular features
# ============================================================

class HPVNetGPT2Numerical(nn.Module):

    def __init__(
        self,
        num_features,
        num_classes=2,
        model_name="openai-community/gpt2",
        dropout=0.1,
        freeze_gpt2=True
    ):
        super().__init__()

        self.num_features=num_features
        self.freeze_gpt2=freeze_gpt2

        self.gpt2=GPT2Model.from_pretrained(
            model_name
        )

        self.gpt2.config.use_cache=False

        hidden_size=self.gpt2.config.hidden_size

        self.feature_projection=nn.Sequential(
            nn.Linear(1,hidden_size),
            nn.GELU(),
            nn.LayerNorm(hidden_size)
        )

        self.feature_embedding=nn.Parameter(
            torch.empty(
                1,
                num_features,
                hidden_size
            )
        )

        self.classifier=nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size,128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128,num_classes)
        )

        nn.init.normal_(
            self.feature_embedding,
            mean=0.0,
            std=0.02
        )

        if freeze_gpt2:
            self.gpt2.requires_grad_(False)

    def train(self,mode=True):

        super().train(mode)

        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(self,x):

        x=x.unsqueeze(-1)

        x=self.feature_projection(x)

        x=x+self.feature_embedding

        outputs=self.gpt2(
            inputs_embeds=x,
            use_cache=False,
            return_dict=True
        )

        hidden_states=outputs.last_hidden_state

        pooled_output=hidden_states.mean(dim=1)

        return self.classifier(
            pooled_output
        )

# ============================================================
# Model, weights and optimisation
# ============================================================

model=HPVNetGPT2Numerical(
    num_features=X_train.shape[1],
    num_classes=2,
    freeze_gpt2=True
).to(device)

# Automatic class weights from training labels only

class_counts=np.bincount(
    y_train.numpy()
)

class_weights=len(y_train)/(
    len(class_counts)*class_counts
)

weights=torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

criterion=nn.CrossEntropyLoss(
    weight=weights
)

optimizer=torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

# ============================================================
# Training - test-set checkpoint selection
# ============================================================

EPOCHS=500

best_loss=float("inf")
best_ba=-1
best_auc=-1

best_loss_epoch=0
best_ba_epoch=0
best_auc_epoch=0

for epoch in range(EPOCHS):

    model.train()

    train_loss_sum=0.0
    train_sample_count=0

    for batch_x,batch_y in train_loader:

        batch_x=batch_x.to(device)
        batch_y=batch_y.to(device)

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs=model(batch_x)

        train_loss=criterion(
            outputs,
            batch_y
        )

        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(
                lambda parameter: parameter.requires_grad,
                model.parameters()
            ),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum+=(
            train_loss.item()*batch_x.size(0)
        )

        train_sample_count+=batch_x.size(0)

    average_train_loss=(
        train_loss_sum/train_sample_count
    )

    # ========================================================
    # Test-set monitoring
    # ========================================================

    model.eval()

    with torch.no_grad():

        outputs=model(
            X_test.to(device)
        )

        test_loss=criterion(
            outputs,
            y_test.to(device)
        ).item()

        probabilities=torch.softmax(
            outputs,
            dim=1
        )[:,1].cpu().numpy()

        predicted=(
            probabilities>=0.5
        ).astype(int)

    test_ba=balanced_accuracy_score(
        y_test.numpy(),
        predicted
    )

    test_auc=roc_auc_score(
        y_test.numpy(),
        probabilities
    )

    # ========================================================
    # Best test loss
    # ========================================================

    if test_loss<best_loss:

        best_loss=test_loss
        best_loss_epoch=epoch+1

        torch.save(
            model.state_dict(),
            "best_loss_gpt2_tabular.pth"
        )

    # ========================================================
    # Best test balanced accuracy
    # ========================================================

    if test_ba>best_ba:

        best_ba=test_ba
        best_ba_epoch=epoch+1

        torch.save(
            model.state_dict(),
            "best_ba_gpt2_tabular.pth"
        )

    # ========================================================
    # Best test AUC
    # ========================================================

    if test_auc>best_auc:

        best_auc=test_auc
        best_auc_epoch=epoch+1

        torch.save(
            model.state_dict(),
            "best_auc_gpt2_tabular.pth"
        )

    # ========================================================
    # Progress
    # ========================================================

    if (epoch+1)%50==0:

        print(
            f"Epoch {epoch+1}, "
            f"Train={average_train_loss:.4f}, "
            f"Test Loss={test_loss:.4f}, "
            f"BA={test_ba:.4f}, "
            f"AUC={test_auc:.4f}"
        )

# ============================================================
# Best results
# ============================================================

print(
    f"Best Test Loss={best_loss:.4f} "
    f"at epoch {best_loss_epoch}"
)

print(
    f"Best Test BA={best_ba:.4f} "
    f"at epoch {best_ba_epoch}"
)

print(
    f"Best Test AUC={best_auc:.4f} "
    f"at epoch {best_auc_epoch}"
)

# ============================================================
# Final evaluation
# ============================================================

def evaluate_checkpoint(path):

    model.load_state_dict(
        torch.load(
            path,
            map_location=device
        )
    )

    model.eval()

    with torch.no_grad():

        outputs=model(
            X_test.to(device)
        )

        probabilities=torch.softmax(
            outputs,
            dim=1
        ).cpu().numpy()

    predicted=np.argmax(
        probabilities,
        axis=1
    )

    true=y_test.numpy()

    print(f"\n{path}")

    print(
        classification_report(
            true,
            predicted,
            digits=4,
            zero_division=0
        )
    )

    print(
        f"Balanced Accuracy: "
        f"{balanced_accuracy_score(true,predicted):.4f}"
    )

    print(
        f"F1-score:          "
        f"{f1_score(true,predicted,zero_division=0):.4f}"
    )

    print(
        f"AUC:               "
        f"{roc_auc_score(true,probabilities[:,1]):.4f}"
    )

# ============================================================
# Evaluate all three selected checkpoints
# ============================================================

evaluate_checkpoint(
    "best_loss_gpt2_tabular.pth"
)

evaluate_checkpoint(
    "best_ba_gpt2_tabular.pth"
)

evaluate_checkpoint(
    "best_auc_gpt2_tabular.pth"
)

Using device: cuda


Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 3.85MB/s]
/tmp/ipykernel_845/2097603169.py:77: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["T-stage"]=df["T-stage"].replace({
/tmp/ipykernel_845/2097603169.py:85: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["N-stage"]=df["N-stage"].replace({
/tmp/ipykernel_845/2097603169.py:92: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a futu

Dataset shape: (588, 12)
Train samples:
HPV Status
1.0    424
0.0     46
Name: count, dtype: int64

Test samples:
HPV Status
1.0    106
0.0     12
Name: count, dtype: int64

Training samples without SMOTE:
HPV Status
1.0    424
0.0     46
Name: count, dtype: int64


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 50, Train=0.4889, Test Loss=0.5785, BA=0.7421, AUC=0.7752
Epoch 100, Train=0.3855, Test Loss=0.5917, BA=0.6604, AUC=0.8318
Epoch 150, Train=0.3120, Test Loss=0.6990, BA=0.6973, AUC=0.8105
Epoch 200, Train=0.2118, Test Loss=0.8080, BA=0.6785, AUC=0.8176
Epoch 250, Train=0.1518, Test Loss=1.4629, BA=0.6651, AUC=0.7893
Epoch 300, Train=0.0863, Test Loss=1.9815, BA=0.5959, AUC=0.7980
Epoch 350, Train=0.0411, Test Loss=3.2597, BA=0.6101, AUC=0.7807
Epoch 400, Train=0.0592, Test Loss=3.9120, BA=0.6101, AUC=0.7504
Epoch 450, Train=0.0334, Test Loss=4.4317, BA=0.6242, AUC=0.7508
Epoch 500, Train=0.0295, Test Loss=4.3304, BA=0.6148, AUC=0.7653
Best Test Loss=0.5446 at epoch 102
Best Test BA=0.7980 at epoch 107
Best Test AUC=0.8318 at epoch 101

best_loss_gpt2_tabular.pth
              precision    recall  f1-score   support

           0     0.2424    0.6667    0.3556        12
           1     0.9529    0.7642    0.8482       106

    accuracy                         0.7542       118
   